# **Import libraries**

In [1]:
!pip install -q streamlit

In [2]:
!pip install pyngrok

In [3]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get('NGROK_KEY'))

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://jordyn-meuni-valeri.ngrok-free.dev" -> "http://localhost:8501"


In [72]:
%%writefile app.py
import streamlit as st
import pandas as pd

Overwriting app.py


# **Contents**

In [73]:
%%writefile -a app.py
st.title('Titanic')

Appending to app.py


## **Home Page**

In [74]:
%%writefile -a app.py

st.header("Home Page")

st.subheader("Titanic Description")

st.write(f"""The sinking of the Titanic is one of the most infamous shipwrecks in history.\n
  On April 15, 1912, during her maiden voyage, the widely considered “unsinkable” RMS Titanic sank after colliding with an iceberg. Unfortunately, there weren’t enough lifeboats for everyone onboard, resulting in the death of 1502 out of 2224 passengers and crew.\n
  While there was some element of luck involved in surviving, it seems some groups of people were more likely to survive than others.""")

st.image('https://upload.wikimedia.org/wikipedia/commons/thumb/6/6e/St%C3%B6wer_Titanic.jpg/1920px-St%C3%B6wer_Titanic.jpg')

data_dict = {
    "Variable": ["survival", "pclass", "sex", "age", "sibsp", "parch", "ticket", "fare", "cabin", "embarked"],
    "Definition": [
        "Survival",
        "Ticket class",
        "Sex",
        "Age in years",
        "# of siblings / spouses aboard the Titanic",
        "# of parents / children aboard the Titanic",
        "Ticket number",
        "Passenger fare",
        "Cabin number",
        "Port of Embarkation"
    ],
    "Key": [
        "0 = No, 1 = Yes",
        "1 = 1st, 2 = 2nd, 3 = 3rd",
        "","","","","","","",
        "C = Cherbourg, Q = Queenstown, S = Southampton"
    ]
}

df_info = pd.DataFrame(data_dict)

st.subheader("Dataset Description")
st.dataframe(df_info)

Appending to app.py


## **Sidebar**

In [75]:
%%writefile -a app.py

st.sidebar.title("Sidebar")
# st.sidebar.write("This is sidebar content.")

# tab1, tab2 = st.tabs(["Parameters", "Tab 2"])

Appending to app.py


## **Parameters Tab**

In [76]:
%%writefile pages/model.py

import streamlit as st
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier

Overwriting pages/model.py


In [77]:
%%writefile -a pages/model.py
df = pd.read_csv('train.csv')

df.drop(['PassengerId','Name','Ticket','Cabin'], axis=1, inplace=True)

Appending to pages/model.py


In [78]:
%%writefile -a pages/model.py
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test  = train_test_split(X, y, random_state=42, test_size=0.2, stratify=y)

# Encode the categorical cols
lbl_enc = LabelEncoder()

X_train['Embarked'] = lbl_enc.fit_transform(X_train['Embarked'])
X_test['Embarked'] = lbl_enc.transform(X_test['Embarked'])

X_train['Sex'] = X_train['Sex'] == 'male'
X_test['Sex'] = X_test['Sex'] == 'male'

# Scale numerical cols
cols_to_scale = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
std_scaler = StandardScaler()

X_train[cols_to_scale] = std_scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = std_scaler.transform(X_test[cols_to_scale])

Appending to pages/model.py


In [79]:
%%writefile -a pages/model.py

st.header("Model Page")
st.latex(r"\Huge \textbf{Fine-Tune\ Your\ Own\ Model}")
st.markdown('The model used is **RandomForestClassifier**.')
st.code('print(Good luck)_', language='python')

# Widgets for parameter insertion
criterion = st.selectbox("Choose the criterion", ["gini", "entropy", "log_loss"])
n_estimators = st.slider('# of estimators', 10, 250)
max_depth = st.slider('max_depth', 2, 15)
min_samples_split = st.slider('min # of samples to split', 2, 10)
min_samples_leaf = st.slider('min # of samples per leaf', 1, 10)
min_impurity_decrease = st.slider('min impurity decrease', 0.0, 1.0)
class_weight = st.selectbox("Choose the class weight", [None, "balanced", "balanced_subsample"])

Appending to pages/model.py


In [80]:
%%writefile -a pages/model.py

st.write(df.head())

Appending to pages/model.py


In [81]:
%%writefile -a pages/model.py

# Container for displaying chosen parameters
with st.container(border=True):
  col1, col2 = st.columns(2)
  with col1:
    st.write('### Parameters')
    st.write('criterion')
    st.write('n_estimators')
    st.write('max_depth')
    st.write('min_samples_split')
    st.write('min_samples_leaf')
    st.write('min_impurity_decrease')
    st.write('class_weight')

  with col2:
    st.write('### Selected Parameters')
    st.write(criterion)
    st.write(n_estimators)
    st.write(max_depth)
    st.write(min_samples_split)
    st.write(min_samples_leaf)
    st.write(min_impurity_decrease)
    st.write(class_weight)

Appending to pages/model.py


In [82]:
%%writefile -a pages/model.py

# Initialize model in session state
if "model" not in st.session_state:
    st.session_state.model = None
    st.session_state.model_performance = 0

with st.container():
  col1, col2 = st.columns(2)
  with col1:
    if st.button('Train'):
      model = RandomForestClassifier(
          criterion=criterion,
          n_estimators=n_estimators,
          max_depth=max_depth,
          min_samples_split=min_samples_split,
          min_samples_leaf=min_samples_leaf,
          min_impurity_decrease=min_impurity_decrease,
          random_state=42)
      model.fit(X_train, y_train)
      st.session_state.model = model
      st.success('Successfully trained')

  with col2:
    if st.button('Test'):
      if st.session_state.model is None:
          st.warning("Train the model first")
      else:
          score = st.session_state.model.score(X_test, y_test)
          st.metric('Score', score, format="%.2f", delta = score - st.session_state.model_performance)
          st.session_state.model_performance = score

Appending to pages/model.py


In [83]:
%%writefile -a pages/model.py

df_tr = pd.concat([X_train, y_train], axis=1)
df_tt = pd.concat([X_test, y_test], axis=1)
df_all = pd.concat([df_tr, df_tt], axis=0)

st.subheader("Charts")

#Correlation Heatmap
st.markdown("### Feature Correlation")

fig1, ax1 = plt.subplots(figsize=(10, 6))
sns.heatmap(df_all.corr(), annot=False, cmap='viridis', ax=ax1)
st.pyplot(fig1)


#Distribution and Confusion Matrix
col1, col2 = st.columns(2)

with col1:
    st.markdown("### Survival Distribution")
    st.bar_chart(
        df_all['Survived'].value_counts(),
        color='violet',
        y_label='# of people',
        x_label='Survived',
        height="stretch")

with col2:
    st.markdown("### Model Performance")

    if st.session_state.model is None:
        st.warning("Train the model first")
    else:
        y_pred = st.session_state.model.predict(X_test)

        fig2, ax2 = plt.subplots()
        cm = confusion_matrix(y_test, y_pred)

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=['Dead', 'Survived']
        )
        disp.plot(ax=ax2)

        st.pyplot(fig2)

Appending to pages/model.py


# **Run streamlit**

In [84]:
!pkill streamlit

!streamlit run app.py &>/content/logs.txt &
